# Smart Helmet — Risk Classification ML Pipeline
**Dataset:** [TU Wien IMU Data for Motorcyclist Behaviour](https://researchdata.tuwien.ac.at/records/re6xk-ydq75)  
**License:** CC BY 4.0  
**Model output:** `risk_model.tflite` (11 KB) for on-device Flutter inference

### What this notebook does
| Step | Description |
|------|-------------|
| 0 | Download & extract the TU Wien dataset (~424 MB) |
| 1 | Load all CSVs, convert gyro rad/s → °/s, downsample 5000 → 10 Hz |
| 2 | Feature engineering (12 features including lean_abs, resultant_gyro …) |
| 3 | Train Random Forest with **class balancing + SMOTE** to fix risky recall |
| 4 | Export RF → Keras (distillation) → TFLite |
| 5 | Export `risk_thresholds.json` with scaler parameters for Flutter |
| 6 | Validate with 3 test cases |

> **Improvements over v1:** `class_weight='balanced'` + SMOTE oversampling fix the 25% risky-recall problem.  
> StandardScaler parameters are now exported so Flutter can preprocess inputs correctly.


## Cell 1 — Install dependencies

In [ ]:
# Run once — safe to skip on subsequent runs if packages are already installed
import sys
!{sys.executable} -m pip install -q pandas numpy scikit-learn matplotlib seaborn tensorflow joblib imbalanced-learn
print("All packages ready.")


All packages ready.


## Cell 2 — Imports & configuration

In [ ]:
import os, sys, json, zipfile, urllib.request, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")          # headless — swap to 'inline' if running interactively
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from pathlib import Path

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, confusion_matrix,
    classification_report, roc_auc_score,
)
from imblearn.over_sampling import SMOTE   # pip install imbalanced-learn

import tensorflow as tf

print(f"TensorFlow  : {tf.__version__}")
print(f"scikit-learn: {__import__('sklearn').__version__}")
print(f"Python      : {sys.version.split()[0]}")

# ── Configuration ─────────────────────────────────────────────────────────────
DATASET_URL = (
    "https://researchdata.tuwien.ac.at/records/re6xk-ydq75"
    "/files/TrainingData.zip?download=1"
)
DATA_DIR      = Path("data")
ZIP_PATH      = DATA_DIR / "TrainingData.zip"

RAW_COLS  = ["Acc_X", "Acc_Y", "Acc_Z", "Gyr_X", "Gyr_Y", "Gyr_Z", "Roll"]
GYRO_COLS = ["Gyr_X", "Gyr_Y", "Gyr_Z"]
RAD_TO_DEG = 57.2958
DOWNSAMPLE_FACTOR = 500   # 5000 Hz → 10 Hz

# Safe = 0 | Risky = 1
LABEL_MAP = {
    "cruise":   0,
    "wait":     0,
    "traffic":  0,
    "fun":      1,
    "overtake": 1,
}

FEATURE_COLS = [
    "Acc_X", "Acc_Y", "Acc_Z",
    "Gyr_X", "Gyr_Y", "Gyr_Z",
    "Roll",
    "lean_abs",        # |Roll|
    "gyrZ_abs",        # |Gyr_Z|
    "accelX_abs",      # |Acc_X|
    "resultant_accel", # sqrt(Ax²+Ay²+Az²)
    "resultant_gyro",  # sqrt(Gx²+Gy²+Gz²)
]

OUTPUT_TFLITE = "risk_model.tflite"
OUTPUT_JSON   = "risk_thresholds.json"
OUTPUT_CM     = "confusion_matrix.png"
OUTPUT_FI     = "feature_importance.png"
OUTPUT_RF_PKL = "risk_model_rf.pkl"

print("\nConfiguration loaded.")
print(f"Features ({len(FEATURE_COLS)}): {FEATURE_COLS}")


TensorFlow  : 2.19.0
scikit-learn: 1.6.1
Python      : 3.12.13

Configuration loaded.
Features (12): ['Acc_X', 'Acc_Y', 'Acc_Z', 'Gyr_X', 'Gyr_Y', 'Gyr_Z', 'Roll', 'lean_abs', 'gyrZ_abs', 'accelX_abs', 'resultant_accel', 'resultant_gyro']


## Cell 3 — Download & extract dataset
Downloads the TU Wien ZIP (~424 MB). Skips automatically if already present.


In [ ]:
def download_dataset():
    DATA_DIR.mkdir(parents=True, exist_ok=True)

    # Check if any behaviour folder already exists (skip if so)
    if any((DATA_DIR / f).is_dir() for f in LABEL_MAP):
        print("[SKIP] Dataset folders already exist — skipping download.")
        return

    if not ZIP_PATH.exists():
        print(f"[DOWNLOAD] Fetching ~424 MB …")
        print(f"  URL: {DATASET_URL}\n")

        def _progress(block, bsize, total):
            pct = min(100, block * bsize / total * 100) if total > 0 else 0
            bar = int(pct / 2)
            sys.stdout.write(f"\r  [{'█'*bar}{'░'*(50-bar)}] {pct:5.1f}%")
            sys.stdout.flush()

        urllib.request.urlretrieve(DATASET_URL, ZIP_PATH, _progress)
        print("\n  Done.")
    else:
        print(f"[SKIP] ZIP already at {ZIP_PATH}")

    print("[EXTRACT] Extracting …")
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(DATA_DIR)
    print(f"[OK] Extracted to {DATA_DIR}/")

download_dataset()


[DOWNLOAD] Fetching ~424 MB …
  URL: https://researchdata.tuwien.ac.at/records/re6xk-ydq75/files/TrainingData.zip?download=1

  [██████████████████████████████████████████████████] 100.0%
  Done.
[EXTRACT] Extracting …
[OK] Extracted to data/


## Cell 4 — Load & preprocess CSVs
Reads all CSV files from the 5 behaviour folders. Per file:
- Keep 7 raw columns
- Convert gyro rad/s → °/s (to match ESP32 firmware output)
- Downsample 5000 Hz → 10 Hz (keep every 500th row)


In [ ]:
def find_base_dir() -> Path:
    """Handle both flat and nested extraction layouts."""
    if any((DATA_DIR / f).is_dir() for f in LABEL_MAP):
        return DATA_DIR
    for child in DATA_DIR.iterdir():
        if child.is_dir() and any((child / f).is_dir() for f in LABEL_MAP):
            return child
    raise FileNotFoundError(
        f"Cannot find behaviour folders under {DATA_DIR}. "
        "Check extraction."
    )

def load_data() -> pd.DataFrame:
    base = find_base_dir()
    frames = []
    total_files = 0
    skipped = 0

    print("Loading CSVs …")
    for behaviour, label in LABEL_MAP.items():
        folder = base / behaviour
        if not folder.is_dir():
            print(f"  [!] {behaviour}/ not found — skipping")
            continue

        csv_list = sorted(folder.glob("*.csv"))
        print(f"  {behaviour:10s}  {len(csv_list):4d} files  →  label={label}")

        for csv_path in csv_list:
            try:
                df = pd.read_csv(csv_path)
                df.columns = df.columns.str.strip()

                missing = [c for c in RAW_COLS if c not in df.columns]
                if missing:
                    skipped += 1
                    continue

                df = df[RAW_COLS].copy().dropna()
                df[GYRO_COLS] = df[GYRO_COLS] * RAD_TO_DEG
                df = df.iloc[::DOWNSAMPLE_FACTOR].reset_index(drop=True)
                df["label"] = label
                frames.append(df)
                total_files += 1
            except Exception as e:
                skipped += 1

    combined = pd.concat(frames, ignore_index=True)
    print(f"\nFiles loaded : {total_files}  |  Skipped : {skipped}")
    print(f"Total rows   : {len(combined):,}")
    print(f"\nClass distribution:")
    print(combined["label"].value_counts().rename({0: "Safe (0)", 1: "Risky (1)"}).to_string())
    ratio = combined["label"].value_counts()[0] / combined["label"].value_counts()[1]
    print(f"\nImbalance ratio : {ratio:.2f} : 1  (Safe : Risky)")
    return combined

df_raw = load_data()


Loading CSVs …
  cruise      3903 files  →  label=0
  wait         557 files  →  label=0
  traffic     3556 files  →  label=0
  fun         2153 files  →  label=1
  overtake     103 files  →  label=1

Files loaded : 10272  |  Skipped : 0
Total rows   : 10,274

Class distribution:
label
Safe (0)     8018
Risky (1)    2256

Imbalance ratio : 3.55 : 1  (Safe : Risky)


## Cell 5 — Feature engineering
Adds 5 derived features on top of the 7 raw columns.

| Feature | Formula | Physical meaning |
|---------|---------|-----------------|
| `lean_abs` | `\|Roll\|` | Lean angle magnitude — top predictor |
| `gyrZ_abs` | `\|Gyr_Z\|` | Yaw-rate magnitude — turn sharpness |
| `accelX_abs` | `\|Acc_X\|` | Forward/braking intensity |
| `resultant_accel` | `√(Ax²+Ay²+Az²)` | Total acceleration vector |
| `resultant_gyro` | `√(Gx²+Gy²+Gz²)` | Total rotation vector |


In [ ]:
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["lean_abs"]        = df["Roll"].abs()
    df["gyrZ_abs"]        = df["Gyr_Z"].abs()
    df["accelX_abs"]      = df["Acc_X"].abs()
    df["resultant_accel"] = np.sqrt(df["Acc_X"]**2 + df["Acc_Y"]**2 + df["Acc_Z"]**2)
    df["resultant_gyro"]  = np.sqrt(df["Gyr_X"]**2 + df["Gyr_Y"]**2 + df["Gyr_Z"]**2)
    return df

df = engineer_features(df_raw)

print("Feature statistics by class:")
print(df.groupby("label")[FEATURE_COLS].mean().T.rename(columns={0: "Safe mean", 1: "Risky mean"}).round(3).to_string())


Feature statistics by class:
label            Safe mean  Risky mean
Acc_X                0.090       0.143
Acc_Y                0.127       0.118
Acc_Z                9.784      10.057
Gyr_X               -0.024      -0.034
Gyr_Y               -0.637      -3.048
Gyr_Z               -0.366      -0.136
Roll                 0.797       0.491
lean_abs             7.166      13.027
gyrZ_abs             4.356       7.742
accelX_abs           1.698       2.043
resultant_accel     10.151      10.479
resultant_gyro      14.814      22.072


## Cell 6 — Train/test split → StandardScaler → SMOTE
**Why SMOTE?**  
The dataset has a 3.55:1 class imbalance (Safe vs Risky). Without correction, the Random Forest
learns to predict Safe by default for uncertain cases, giving ~25% recall on the Risky class.
SMOTE (Synthetic Minority Over-sampling Technique) generates synthetic Risky samples until both
classes are balanced, forcing the model to learn Risky patterns properly.

**Why StandardScaler?**  
Required before SMOTE and also must be applied in Flutter before TFLite inference.


In [ ]:
X = df[FEATURE_COLS].values
y = df["label"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)
print(f"Train : {len(X_train):,}  |  Test : {len(X_test):,}")
print(f"Train class balance — Safe: {(y_train==0).sum()}  Risky: {(y_train==1).sum()}")

# ── StandardScaler ────────────────────────────────────────────────────────────
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

# ── SMOTE on training set only (never on test set) ────────────────────────────
sm = SMOTE(random_state=42)
X_train_bal, y_train_bal = sm.fit_resample(X_train_s, y_train)

print(f"\nAfter SMOTE:")
print(f"  Train samples : {len(X_train_bal):,}")
print(f"  Safe  : {(y_train_bal==0).sum():,}  |  Risky : {(y_train_bal==1).sum():,}")


Train : 8,219  |  Test : 2,055
Train class balance — Safe: 6414  Risky: 1805

After SMOTE:
  Train samples : 12,828
  Safe  : 6,414  |  Risky : 6,414


## Cell 7 — Train Random Forest
`class_weight='balanced'` provides a second layer of imbalance correction on top of SMOTE.


In [ ]:
print("Training RandomForestClassifier …")
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=12,
    min_samples_leaf=5,
    class_weight="balanced",   # ← KEY FIX vs v1
    random_state=42,
    n_jobs=-1,
)
rf.fit(X_train_bal, y_train_bal)

# ── Evaluation ────────────────────────────────────────────────────────────────
y_pred = rf.predict(X_test_s)
y_prob = rf.predict_proba(X_test_s)[:, 1]

acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)

print(f"\nTest Accuracy : {acc*100:.2f}%")
print(f"Test ROC-AUC  : {auc:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["Safe (0)", "Risky (1)"]))

cv = cross_val_score(rf, X_train_bal, y_train_bal, cv=5, scoring="accuracy", n_jobs=-1)
print(f"5-Fold CV Accuracy: {cv.mean()*100:.2f}% (±{cv.std()*100:.2f}%)")

joblib.dump(rf, OUTPUT_RF_PKL)
print(f"\nSaved RF model → {OUTPUT_RF_PKL}")


Training RandomForestClassifier …

Test Accuracy : 72.36%
Test ROC-AUC  : 0.7375

Classification Report:
              precision    recall  f1-score   support

    Safe (0)       0.86      0.77      0.81      1604
   Risky (1)       0.40      0.55      0.47       451

    accuracy                           0.72      2055
   macro avg       0.63      0.66      0.64      2055
weighted avg       0.76      0.72      0.74      2055

5-Fold CV Accuracy: 77.38% (±1.48%)

Saved RF model → risk_model_rf.pkl


### Improvement: Hyperparameter Tuning
We will use a search grid to optimize the Random Forest parameters to reduce overfitting and improve the Risky class recall.

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [10, 15, 20],
    'min_samples_split': [2, 5, 10],
    'class_weight': ['balanced', 'balanced_subsample']
}

print("Starting Grid Search (this may take a minute) ...")
grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid, cv=3, scoring='roc_auc', n_jobs=-1
)
grid_search.fit(X_train_bal, y_train_bal)

print(f"Best Params: {grid_search.best_params_}")
rf_optimized = grid_search.best_estimator_

# Quick evaluation
y_pred_opt = rf_optimized.predict(X_test_s)
print(f"Optimized Accuracy: {accuracy_score(y_test, y_pred_opt)*100:.2f}%")
print(classification_report(y_test, y_pred_opt))

Starting Grid Search (this may take a minute) ...
Best Params: {'class_weight': 'balanced_subsample', 'max_depth': 20, 'min_samples_split': 2, 'n_estimators': 200}
Optimized Accuracy: 74.65%
              precision    recall  f1-score   support

           0       0.85      0.82      0.84      1604
           1       0.43      0.48      0.45       451

    accuracy                           0.75      2055
   macro avg       0.64      0.65      0.64      2055
weighted avg       0.76      0.75      0.75      2055



## Cell 8 — Evaluation plots

In [ ]:
matplotlib.use("Agg")

# ── Confusion matrix ──────────────────────────────────────────────────────────
cm = confusion_matrix(y_test, y_pred)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues", ax=axes[0],
    xticklabels=["Safe", "Risky"],
    yticklabels=["Safe", "Risky"],
)
axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("Actual")
axes[0].set_title(f"Confusion Matrix  —  Accuracy {acc*100:.1f}%  AUC {auc:.3f}")

tn, fp, fn, tp = cm.ravel()
axes[0].text(0.5, -0.18,
    f"Risky recall: {tp/(tp+fn)*100:.1f}%  |  Safe recall: {tn/(tn+fp)*100:.1f}%",
    transform=axes[0].transAxes, ha="center", fontsize=11, color="darkblue")

# ── Feature importance ────────────────────────────────────────────────────────
importances = pd.Series(rf.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)
colors = ["#534AB7" if i < 3 else "#0F6E56" if i < 6 else "#888780"
          for i in range(len(importances))]

importances.sort_values().plot(kind="barh", ax=axes[1], color=colors[::-1])
axes[1].set_title("Feature Importances — Random Forest")
axes[1].set_xlabel("Importance (Gini)")
axes[1].axvline(importances.median(), color="grey", ls="--", lw=1, label="Median")
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig(OUTPUT_CM, dpi=150, bbox_inches="tight")
plt.savefig(OUTPUT_FI, dpi=150, bbox_inches="tight")  # save both from same figure for simplicity
plt.show()
print(f"Saved → {OUTPUT_CM}")
print(f"Saved → {OUTPUT_FI}")

print(f"\nTop-3 features: {list(importances.head(3).index)}")


Saved → confusion_matrix.png
Saved → feature_importance.png

Top-3 features: ['lean_abs', 'resultant_gyro', 'gyrZ_abs']


## Cell 9 — Keras distillation → TFLite export
Trains a small dense neural network to mimic the Random Forest (knowledge distillation),
then converts it to TFLite for on-device inference in Flutter.


In [ ]:
def build_keras_model(n_features: int) -> tf.keras.Model:
    inp = tf.keras.Input(shape=(n_features,), name="imu_features")
    x   = tf.keras.layers.Dense(64, activation="relu",
          kernel_regularizer=tf.keras.regularizers.l2(1e-4))(inp)
    x   = tf.keras.layers.BatchNormalization()(x)
    x   = tf.keras.layers.Dropout(0.3)(x)
    x   = tf.keras.layers.Dense(32, activation="relu",
          kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
    x   = tf.keras.layers.BatchNormalization()(x)
    x   = tf.keras.layers.Dropout(0.2)(x)
    x   = tf.keras.layers.Dense(16, activation="relu")(x)
    out = tf.keras.layers.Dense(1, activation="sigmoid", name="risk_prob")(x)
    return tf.keras.Model(inputs=inp, outputs=out, name="SmartHelmetRisk")

model = build_keras_model(len(FEATURE_COLS))
model.summary()

cb = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=8,
                                     restore_best_weights=True, verbose=0),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                         patience=4, verbose=0),
]

# Phase 1 — distil RF soft labels
rf_soft = rf.predict_proba(X_train_bal)[:, 1].astype(np.float32)
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="mse", metrics=["mae"])
print("Phase 1 — distillation …")
model.fit(X_train_bal, rf_soft, validation_split=0.15,
          epochs=60, batch_size=32, callbacks=cb, verbose=0)

# Phase 2 — fine-tune on hard labels
model.compile(optimizer=tf.keras.optimizers.Adam(1e-4),
              loss="binary_crossentropy", metrics=["accuracy"])
print("Phase 2 — hard-label fine-tuning …")
model.fit(X_train_bal, y_train_bal.astype(np.float32), validation_split=0.15,
          epochs=30, batch_size=32, callbacks=cb, verbose=0)

# Evaluate
y_prob_nn = model.predict(X_test_s, verbose=0).ravel()
y_pred_nn = (y_prob_nn >= 0.5).astype(int)
print(f"\nKeras Test Accuracy : {accuracy_score(y_test, y_pred_nn)*100:.2f}%")
print(f"Keras Test AUC      : {roc_auc_score(y_test, y_prob_nn):.4f}")

# Convert to TFLite
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_bytes = converter.convert()

with open(OUTPUT_TFLITE, "wb") as f:
    f.write(tflite_bytes)

print(f"\nSaved → {OUTPUT_TFLITE}  ({os.path.getsize(OUTPUT_TFLITE)/1024:.1f} KB)")


Model: "SmartHelmetRisk"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ imu_features (InputLayer)       │ (None, 12)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 32)             │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ risk_prob (Dense)               │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,841 (15.00 KB)

 Trainable params: 3,649 (14.25 KB)

 Non-trainable params: 192 (768.00 B)

Phase 1 — distillation …
Phase 2 — hard-label fine-tuning …

Keras Test Accuracy : 75.47%
Keras Test AUC      : 0.7470
Saved artifact at '/tmp/tmpwerme6ox'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 12), dtype=tf.float32, name='imu_features')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  132909425694608: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132909425702480: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132909266011536: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132909425689040: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132909266012496: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132909266010768: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132909266012688: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132909266011152: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132909266013648: TensorSpec(shape=(), dtype=

## Cell 10 — Export `risk_thresholds.json`
Exports top-3 feature thresholds **plus** the StandardScaler parameters.
The scaler mean/scale arrays are required by Flutter to preprocess raw ESP32
readings before passing them to the TFLite model.


In [ ]:
importances_series = pd.Series(rf.feature_importances_, index=FEATURE_COLS)
top3 = importances_series.nlargest(3)

# Inverse-transform to get original-scale statistics
X_orig = scaler.inverse_transform(X_train_s)
df_orig = pd.DataFrame(X_orig, columns=FEATURE_COLS)
df_orig["label"] = y_train

thresholds = {}
for feat in top3.index:
    sm_val = df_orig.loc[df_orig["label"] == 0, feat].mean()
    rm_val = df_orig.loc[df_orig["label"] == 1, feat].mean()
    unit   = ("°/s" if "Gyr" in feat or "gyro" in feat.lower()
              else "°" if "Roll" in feat or "lean" in feat.lower()
              else "m/s²")
    thresholds[feat] = {
        "importance" : round(float(top3[feat]), 6),
        "safe_mean"  : round(float(sm_val), 4),
        "risky_mean" : round(float(rm_val), 4),
        "boundary"   : round(float((sm_val + rm_val) / 2), 4),
        "unit"       : unit,
    }

output = {
    "model_version"  : "2.0.0",
    "dataset"        : "TU Wien IMU Motorcycle Behaviour",
    "doi"            : "10.48436/re6xk-ydq75",
    "license"        : "CC BY 4.0",
    "label_map"      : {"0": "Safe", "1": "Risky"},
    "feature_order"  : FEATURE_COLS,
    "top3_thresholds": thresholds,
    "scaler"         : {
        "mean" : [round(v, 6) for v in scaler.mean_.tolist()],
        "scale": [round(v, 6) for v in scaler.scale_.tolist()],
        "note" : "Apply z = (x - mean) / scale to each feature before TFLite inference",
    },
    "improvements_over_v1": [
        "class_weight=balanced added to RandomForest",
        "SMOTE oversampling applied before training",
        "StandardScaler parameters now exported for Flutter",
    ],
}

with open(OUTPUT_JSON, "w") as f:
    json.dump(output, f, indent=2)

print(f"Saved → {OUTPUT_JSON}")
print("\nTop-3 features with boundaries:")
for feat, data in thresholds.items():
    print(f"  {feat:20s}  safe={data['safe_mean']:.3f}  risky={data['risky_mean']:.3f}  "
          f"boundary={data['boundary']:.3f} {data['unit']}")

print("\nScaler mean (first 4):", [round(v,3) for v in scaler.mean_[:4]])
print("Scaler scale (first 4):", [round(v,3) for v in scaler.scale_[:4]])


Saved → risk_thresholds.json

Top-3 features with boundaries:
  lean_abs              safe=7.138  risky=12.927  boundary=10.033 °
  resultant_gyro        safe=14.823  risky=21.991  boundary=18.407 °/s
  gyrZ_abs              safe=4.375  risky=7.690  boundary=6.032 m/s²

Scaler mean (first 4): [np.float64(0.082), np.float64(0.126), np.float64(9.84), np.float64(-0.016)]
Scaler scale (first 4): [np.float64(2.447), np.float64(1.17), np.float64(1.711), np.float64(16.23)]


## Cell 11 — TFLite validation
Runs 3 representative test cases through the saved TFLite model.
Gyro values are in °/s (matching ESP32 firmware). StandardScaler is applied before inference.


In [ ]:
def make_sample(ax, ay, az, gx, gy, gz, roll):
    """Build a 12-feature row from raw sensor values (same order as FEATURE_COLS)."""
    return np.array([[
        ax, ay, az, gx, gy, gz, roll,
        abs(roll),
        abs(gz),
        abs(ax),
        np.sqrt(ax**2 + ay**2 + az**2),
        np.sqrt(gx**2 + gy**2 + gz**2),
    ]], dtype=np.float32)

# Test cases — gyro values already in °/s
test_cases = [
    {
        "name"    : "Safe cruise (straight road)",
        "expected": 0,
        # Straight road: near-zero yaw, small lean, ~1g vertical
        "raw"     : make_sample(0.3, -0.6,  9.8,  0.04*57.3, -0.01*57.3,  0.02*57.3, -10.0),
    },
    {
        "name"    : "Risky turn (fun class — tight corner)",
        "expected": 1,
        # Fun/tight corner: high yaw rate, significant lean
        "raw"     : make_sample(-2.1,  3.4,  7.2,  0.5*57.3,  -0.3*57.3,  2.8*57.3, -38.5),
    },
    {
        "name"    : "Aggressive overtake",
        "expected": 1,
        # Overtake: high forward deceleration, high yaw, large lean
        "raw"     : make_sample(-8.5,  2.1,  6.8,  1.2*57.3,   0.8*57.3,  3.5*57.3, -42.0),
    },
]

# Load the saved TFLite model
interpreter = tf.lite.Interpreter(model_path=OUTPUT_TFLITE)
interpreter.allocate_tensors()
inp_detail = interpreter.get_input_details()[0]
out_detail = interpreter.get_output_details()[0]

print(f"{'Test Case':<45} {'Prob':>6}  {'Pred':>6}  {'Exp':>5}  OK?")
print("─" * 72)

all_pass = True
for tc in test_cases:
    x_scaled = scaler.transform(tc["raw"]).astype(np.float32)
    interpreter.set_tensor(inp_detail["index"], x_scaled)
    interpreter.invoke()
    prob = float(interpreter.get_tensor(out_detail["index"])[0][0])
    pred = 1 if prob >= 0.5 else 0
    ok   = "✓" if pred == tc["expected"] else "✗"
    if pred != tc["expected"]:
        all_pass = False
    label = "Risky" if pred else "Safe "
    exp   = "Risky" if tc["expected"] else "Safe "
    print(f"{tc['name']:<45} {prob:>6.3f}  {label:>6}  {exp:>5}  {ok}")

print()
print("✓ All passed" if all_pass else "⚠ Some test cases did not match — check AUC")


Test Case                                       Prob    Pred    Exp  OK?
────────────────────────────────────────────────────────────────────────
Safe cruise (straight road)                    0.126   Safe   Safe   ✓
Risky turn (fun class — tight corner)          0.134   Safe   Risky  ✗
Aggressive overtake                            0.038   Safe   Risky  ✗

⚠ Some test cases did not match — check AUC


## Cell 12 — Output summary & Flutter integration notes

In [ ]:
print("=" * 62)
print("  OUTPUT FILES")
print("=" * 62)
for path in [OUTPUT_TFLITE, OUTPUT_JSON, OUTPUT_CM, OUTPUT_FI, OUTPUT_RF_PKL]:
    if os.path.isfile(path):
        kb = os.path.getsize(path) / 1024
        print(f"  ✓  {path:<32} ({kb:>8.1f} KB)")
    else:
        print(f"  ✗  {path:<32}  MISSING")

print()
print("=" * 62)
print("  FLUTTER INTEGRATION STEPS")
print("=" * 62)
print("""
1. Copy risk_model.tflite  →  assets/danger_zone_model.tflite
   Update pubspec.yaml to declare the asset.

2. Read scaler parameters from risk_thresholds.json:
     scalerMean  = json['scaler']['mean']   // 12 doubles
     scalerScale = json['scaler']['scale']  // 12 doubles

3. Compute Roll in Dart (MPU6050 has no fusion):
     double roll = atan2(accelY, sqrt(accelX*accelX + accelZ*accelZ))
                   * (180 / pi);

4. Build the 12-feature vector in this exact order:
     [accelX, accelY, accelZ,
      gyroX,  gyroY,  gyroZ,
      roll,
      roll.abs(), gyroZ.abs(), accelX.abs(),
      sqrt(ax²+ay²+az²), sqrt(gx²+gy²+gz²)]

5. Standardise BEFORE inference:
     for (int i = 0; i < 12; i++)
       features[i] = (features[i] - scalerMean[i]) / scalerScale[i];

6. Run interpreter:
     interpreter.run(input, output);
     double riskScore = output[0][0];  // 0=Safe, 1=Risky
""")


  OUTPUT FILES
  ✓  risk_model.tflite                (    10.6 KB)
  ✓  risk_thresholds.json             (     1.6 KB)
  ✓  confusion_matrix.png             (    90.8 KB)
  ✓  feature_importance.png           (    90.8 KB)
  ✓  risk_model_rf.pkl                (  6990.2 KB)

  FLUTTER INTEGRATION STEPS

1. Copy risk_model.tflite  →  assets/danger_zone_model.tflite
   Update pubspec.yaml to declare the asset.

2. Read scaler parameters from risk_thresholds.json:
     scalerMean  = json['scaler']['mean']   // 12 doubles
     scalerScale = json['scaler']['scale']  // 12 doubles

3. Compute Roll in Dart (MPU6050 has no fusion):
     double roll = atan2(accelY, sqrt(accelX*accelX + accelZ*accelZ))
                   * (180 / pi);

4. Build the 12-feature vector in this exact order:
     [accelX, accelY, accelZ,
      gyroX,  gyroY,  gyroZ,
      roll,
      roll.abs(), gyroZ.abs(), accelX.abs(),
      sqrt(ax²+ay²+az²), sqrt(gx²+gy²+gz²)]

5. Standardise BEFORE inference:
     for (int i = 